# Text Preprocessing

In [40]:
!pip install emoji Sastrawi pyarrow nltk wordcloud tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 1.3 MB/s eta 0:00:00


In [41]:
import pandas as pd
import numpy as np
import re
import time
import json
import os
from collections import Counter
import emoji as emoji_lib
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

pd.set_option('display.max_colwidth', 100)

df = pd.read_parquet('../data/interim/clean_comments_kopdes.parquet')
print(f'Baris input: {len(df):,}')
df[['video_id', 'comment']].head(3)

Baris input: 120,707


,video_id,comment
0,7526458489577737488,Beliau bilang untung nya 2 m ?
1,7526458489577737488,Makasih Mas
2,7526458489577737488,Gimana untung nya 2 m..barang yg di jual ajh ga ada 1 m 😁


## Case Folding

In [42]:
def case_folding(text: str) -> str:
    return text.lower()

df['text_step1_casefold'] = df['comment'].astype(str).apply(case_folding)
df[['comment', 'text_step1_casefold']].head(3)

,comment,text_step1_casefold
0,Beliau bilang untung nya 2 m ?,beliau bilang untung nya 2 m ?
1,Makasih Mas,makasih mas
2,Gimana untung nya 2 m..barang yg di jual ajh ga ada 1 m 😁,gimana untung nya 2 m..barang yg di jual ajh ga ada 1 m 😁


## Remove URL, Mention, Hashtag, Sisa Tag Sistem

In [43]:
def remove_url(text: str) -> str:
    return re.sub(r'https?://\S+|www\.\S+', ' ', text)

def remove_mention(text: str) -> str:
    return re.sub(r'@\w+', ' ', text)

def remove_hashtag(text: str) -> str:
    return re.sub(r'#\w+', ' ', text)

def remove_system_tags(text: str) -> str:
    return re.sub(r'\[sticker\]', ' ', text, flags=re.IGNORECASE)

n_url = df['text_step1_casefold'].str.contains(r'https?://|www\.', regex=True).sum()
n_mention = df['text_step1_casefold'].str.contains(r'@\w+', regex=True).sum()
n_hashtag = df['text_step1_casefold'].str.contains(r'#\w+', regex=True).sum()
print(f'Baris mengandung URL: {n_url:,} | mention: {n_mention:,} | hashtag: {n_hashtag:,}')

df['text_step2_cleaned'] = (
    df['text_step1_casefold']
    .apply(remove_url).apply(remove_mention).apply(remove_hashtag).apply(remove_system_tags)
)
df[['text_step1_casefold', 'text_step2_cleaned']].head(3)

Baris mengandung URL: 29 | mention: 2,817 | hashtag: 106


,text_step1_casefold,text_step2_cleaned
0,beliau bilang untung nya 2 m ?,beliau bilang untung nya 2 m ?
1,makasih mas,makasih mas
2,gimana untung nya 2 m..barang yg di jual ajh ga ada 1 m 😁,gimana untung nya 2 m..barang yg di jual ajh ga ada 1 m 😁


## Remove Emoji

In [44]:
def remove_emoji(text: str) -> str:
    return emoji_lib.replace_emoji(text, replace=' ')

df['text_step3_noemoji'] = df['text_step2_cleaned'].apply(remove_emoji)
df[['text_step2_cleaned', 'text_step3_noemoji']].head(3)

,text_step2_cleaned,text_step3_noemoji
0,beliau bilang untung nya 2 m ?,beliau bilang untung nya 2 m ?
1,makasih mas,makasih mas
2,gimana untung nya 2 m..barang yg di jual ajh ga ada 1 m 😁,gimana untung nya 2 m..barang yg di jual ajh ga ada 1 m


## Remove Number

In [45]:
def remove_number(text: str) -> str:
    return re.sub(r'\d+', ' ', text)

df['text_step4_nonum'] = df['text_step3_noemoji'].apply(remove_number)
df[['text_step3_noemoji', 'text_step4_nonum']].head(3)

,text_step3_noemoji,text_step4_nonum
0,beliau bilang untung nya 2 m ?,beliau bilang untung nya m ?
1,makasih mas,makasih mas
2,gimana untung nya 2 m..barang yg di jual ajh ga ada 1 m,gimana untung nya m..barang yg di jual ajh ga ada m


## Remove Punctuation + Normalisasi Elongasi Huruf

In [46]:
def remove_punctuation(text: str) -> str:
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'_', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def normalize_elongation(text: str) -> str:
    return re.sub(r'(.)\1{2,}', r'\1\1', text)

df['text_step5_clean'] = df['text_step4_nonum'].apply(remove_punctuation).apply(normalize_elongation)
df[['comment', 'text_step5_clean']].head(5)

,comment,text_step5_clean
0,Beliau bilang untung nya 2 m ?,beliau bilang untung nya m
1,Makasih Mas,makasih mas
2,Gimana untung nya 2 m..barang yg di jual ajh ga ada 1 m 😁,gimana untung nya m barang yg di jual ajh ga ada m
3,[Sticker] dibilang Makasih Mas udah 2M itu,dibilang makasih mas udah m itu
4,"Kalo niat membantu UMKM, harusnya koprasi merah putih dibuka dalam bentuk warehouse (gudang), bu...",kalo niat membantu umkm harusnya koprasi merah putih dibuka dalam bentuk warehouse gudang bukan ...


In [47]:
n_before = len(df)
df = df[df['text_step5_clean'].str.strip() != ''].reset_index(drop=True)
print(f'Drop residu kosong pasca-cleaning: {n_before:,} -> {len(df):,} (-{n_before-len(df):,})')
print('(baris ini adalah sisa mention/URL/hashtag/emoji murni tanpa kata lain, terdokumentasi)')

Drop residu kosong pasca-cleaning: 120,707 -> 119,384 (-1,323)
(baris ini adalah sisa mention/URL/hashtag/emoji murni tanpa kata lain, terdokumentasi)


## Tokenization

In [48]:
def tokenize(text: str) -> list:
    return text.split()

df['tokens'] = df['text_step5_clean'].apply(tokenize)
df[['text_step5_clean', 'tokens']].head(5)

,text_step5_clean,tokens
0,beliau bilang untung nya m,"[beliau, bilang, untung, nya, m]"
1,makasih mas,"[makasih, mas]"
2,gimana untung nya m barang yg di jual ajh ga ada m,"[gimana, untung, nya, m, barang, yg, di, jual, ajh, ga, ada, m]"
3,dibilang makasih mas udah m itu,"[dibilang, makasih, mas, udah, m, itu]"
4,kalo niat membantu umkm harusnya koprasi merah putih dibuka dalam bentuk warehouse gudang bukan ...,"[kalo, niat, membantu, umkm, harusnya, koprasi, merah, putih, dibuka, dalam, bentuk, warehouse, ..."


## Stopword Removal

In [49]:
stopword_factory = StopWordRemoverFactory()
stopwords_formal = set(stopword_factory.get_stop_words())

stopwords_colloquial = {
    'yg', 'ga', 'gak', 'gk', 'nggak', 'ngga', 'nya', 'dgn', 'dg', 'aja', 'sih', 'deh', 'kok',
    'dong', 'donk', 'banget', 'bgt', 'emang', 'emg', 'udah', 'udh', 'dah', 'nih', 'tuh', 'kan',
    'ya', 'yah', 'loh', 'lho', 'lah', 'kah', 'pun', 'juga', 'jg', 'sm', 'sama', 'trus', 'terus',
    'krn', 'karna', 'utk', 'sdh', 'blm', 'belom', 'gitu', 'gt', 'gini', 'gni', 'klo', 'kalo',
    'gmn', 'gimana', 'org', 'orang', 'sy', 'saya', 'kalian', 'kmu', 'kamu', 'lu', 'lo', 'gue',
    'gw', 'ni', 'si', 'nder', 'wkwk', 'wkwkwk', 'haha', 'hehe',
}
all_stopwords = stopwords_formal | stopwords_colloquial
print(f'Total stopword: {len(all_stopwords):,}')

def remove_stopwords(tokens: list) -> list:
    return [t for t in tokens if t not in all_stopwords and len(t) > 1]

t0 = time.time()
df['tokens_nostop'] = df['tokens'].apply(remove_stopwords)
print(f'Waktu stopword removal: {time.time()-t0:.2f}s')
df[['tokens', 'tokens_nostop']].head(5)

Total stopword: 187
Waktu stopword removal: 1.16s


,tokens,tokens_nostop
0,"[beliau, bilang, untung, nya, m]","[beliau, bilang, untung]"
1,"[makasih, mas]","[makasih, mas]"
2,"[gimana, untung, nya, m, barang, yg, di, jual, ajh, ga, ada, m]","[untung, barang, jual, ajh]"
3,"[dibilang, makasih, mas, udah, m, itu]","[dibilang, makasih, mas]"
4,"[kalo, niat, membantu, umkm, harusnya, koprasi, merah, putih, dibuka, dalam, bentuk, warehouse, ...","[niat, membantu, umkm, harusnya, koprasi, merah, putih, dibuka, bentuk, warehouse, gudang, bukan..."


## Stemming Bahasa Indonesia (Sastrawi, Frequency-Thresholded + Cached)

In [50]:
FREQ_THRESHOLD = 3
CACHE_PATH = '../scripts/stem_cache_checkpoint.json'

counter = Counter()
for toks in df['tokens_nostop']:
    counter.update(toks)
total_occ = sum(counter.values())
tokens_to_stem = [tok for tok, c in counter.items() if c >= FREQ_THRESHOLD]
occ_covered = sum(counter[t] for t in tokens_to_stem)

print(f'Token unik total          : {len(counter):,}')
print(f'Token di-stem (freq>={FREQ_THRESHOLD})   : {len(tokens_to_stem):,} ({len(tokens_to_stem)/len(counter)*100:.1f}% vocab)')
print(f'Cakupan occurrence         : {occ_covered/total_occ*100:.2f}% dari seluruh kemunculan token')

factory = StemmerFactory()
stemmer = factory.create_stemmer()

os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)

stem_cache = {}
if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH) as f:
        stem_cache = json.load(f)
    print(f'Loaded {len(stem_cache):,} cached stems dari run sebelumnya.')

t0 = time.time()
n_new = 0
for tok in tokens_to_stem:
    if tok not in stem_cache:
        stem_cache[tok] = stemmer.stem(tok)
        n_new += 1
print(f'Token baru di-stem live: {n_new:,} (sisanya cache hit) dalam {time.time()-t0:.2f}s')

with open(CACHE_PATH, 'w') as f:
    json.dump(stem_cache, f)

for w in ['pemerintahan', 'anggarannya', 'membantu', 'dibilang', 'menyediakan']:
    if w in stem_cache:
        print(f'  {w:20s} -> {stem_cache[w]}')

Token unik total          : 45,024
Token di-stem (freq>=3)   : 14,436 (32.1% vocab)
Cakupan occurrence         : 95.37% dari seluruh kemunculan token
Token baru di-stem live: 14,436 (sisanya cache hit) dalam 1148.17s
  pemerintahan         -> perintah
  anggarannya          -> anggar
  membantu             -> bantu
  dibilang             -> bilang
  menyediakan          -> sedia


In [51]:
def stem_tokens(tokens: list) -> list:
    return [stem_cache.get(t, t) for t in tokens]

t0 = time.time()
df['tokens_stemmed'] = df['tokens_nostop'].apply(stem_tokens)
print(f'Waktu mapping stemming ke seluruh baris: {time.time()-t0:.2f}s')

df['tokens_stemmed'] = df['tokens_stemmed'].apply(lambda toks: [t for t in toks if t.strip() != ''])
df['text_final'] = df['tokens_stemmed'].apply(lambda toks: ' '.join(toks))

df[['comment', 'tokens_stemmed', 'text_final']].head(5)

Waktu mapping stemming ke seluruh baris: 0.20s


,comment,tokens_stemmed,text_final
0,Beliau bilang untung nya 2 m ?,"[beliau, bilang, untung]",beliau bilang untung
1,Makasih Mas,"[makasih, mas]",makasih mas
2,Gimana untung nya 2 m..barang yg di jual ajh ga ada 1 m 😁,"[untung, barang, jual, ajh]",untung barang jual ajh
3,[Sticker] dibilang Makasih Mas udah 2M itu,"[bilang, makasih, mas]",bilang makasih mas
4,"Kalo niat membantu UMKM, harusnya koprasi merah putih dibuka dalam bentuk warehouse (gudang), bu...","[niat, bantu, umkm, harus, koprasi, merah, putih, buka, bentuk, warehouse, gudang, bukan, bentuk...",niat bantu umkm harus koprasi merah putih buka bentuk warehouse gudang bukan bentuk toko jadi be...


## Validasi Akhir Preprocessing

In [52]:
n_before_final = len(df)
empty_final = df['text_final'].str.strip() == ''
print(f'Baris tanpa token tersisa setelah stopword removal + stemming: {empty_final.sum():,}')
if empty_final.sum() > 0:
    print('Contoh (comment asli):', df.loc[empty_final, 'comment'].head(5).tolist())

df_final = df[~empty_final].reset_index(drop=True)
print(f'Baris sebelum drop token-kosong : {n_before_final:,}')
print(f'Baris final preprocessing        : {len(df_final):,}')

token_counts = df_final['tokens_stemmed'].apply(len)
print()
print('Distribusi jumlah token per komentar:')
print(token_counts.describe())

assert len(df_final) >= 5000, 'Data di bawah syarat minimum 5000 setelah preprocessing!'
assert df_final['text_final'].isnull().sum() == 0
assert (df_final['text_final'].str.strip() == '').sum() == 0
print()
print('VALIDASI PASSED: tidak ada text_final kosong, memenuhi minimum 5.000 baris.')

Baris tanpa token tersisa setelah stopword removal + stemming: 606
Contoh (comment asli): ['ia😂', '😂😂0m0n9 k0$0n9😩😅😅😩😩', 'ini lagi', 'haha', '@allriseyellow kenapa e']
Baris sebelum drop token-kosong : 119,384
Baris final preprocessing        : 118,778

Distribusi jumlah token per komentar:
count    118778.000000
mean          6.613354
std           6.971723
min           1.000000
25%           3.000000
50%           5.000000
75%           8.000000
max         209.000000
Name: tokens_stemmed, dtype: float64

VALIDASI PASSED: tidak ada text_final kosong, memenuhi minimum 5.000 baris.


## Simpan Hasil Preprocessing

In [53]:
cols_to_keep = [
    'video_id', 'comment_id', 'parent_comment_id', 'level', 'username', 'nickname',
    'create_time', 'comment', 'text_final', 'tokens_stemmed'
]
df_out = df_final[cols_to_keep].copy()
df_out = df_out.rename(columns={'tokens_stemmed': 'tokens'})
df_out_parquet = df_out.copy()
df_out_parquet['tokens'] = df_out_parquet['tokens'].apply(lambda x: ' '.join(x))

os.makedirs('../data/interim', exist_ok=True)
df_out_parquet.to_parquet('../data/interim/preprocessed_comments_kopdes.parquet', index=False)
df_out_parquet.to_csv('../data/interim/preprocessed_comments_kopdes.csv', index=False)

preprocessing_summary = {
    'n_rows_input': int(n_before),
    'n_rows_output': int(len(df_out)),
    'n_unique_tokens_total': len(counter),
    'n_unique_tokens_stemmed': len(tokens_to_stem),
    'freq_threshold_for_stemming': FREQ_THRESHOLD,
    'pct_occurrence_coverage_stemmed': round(occ_covered/total_occ*100, 2),
    'avg_tokens_per_comment': float(token_counts.mean()),
    'median_tokens_per_comment': float(token_counts.median()),
    'stopwords_total': len(all_stopwords),
    'steps_applied': [
        'case_folding', 'remove_url', 'remove_mention', 'remove_hashtag', 'remove_system_tags',
        'remove_emoji', 'remove_number', 'remove_punctuation', 'normalize_elongation',
        'tokenization', 'stopword_removal (formal+colloquial)',
        'stemming (Sastrawi, frequency-thresholded + cached)'
    ]
}
with open('../reports/preprocessing_summary.json', 'w') as f:
    json.dump(preprocessing_summary, f, indent=2)

print(json.dumps(preprocessing_summary, indent=2))
print()
print(f'File tersimpan: data/interim/preprocessed_comments_kopdes.parquet ({len(df_out):,} baris)')

{
  "n_rows_input": 120707,
  "n_rows_output": 118778,
  "n_unique_tokens_total": 45024,
  "n_unique_tokens_stemmed": 14436,
  "freq_threshold_for_stemming": 3,
  "pct_occurrence_coverage_stemmed": 95.37,
  "avg_tokens_per_comment": 6.613354324874977,
  "median_tokens_per_comment": 5.0,
  "stopwords_total": 187,
  "steps_applied": [
    "case_folding",
    "remove_url",
    "remove_mention",
    "remove_hashtag",
    "remove_system_tags",
    "remove_emoji",
    "remove_number",
    "remove_punctuation",
    "normalize_elongation",
    "tokenization",
    "stopword_removal (formal+colloquial)",
    "stemming (Sastrawi, frequency-thresholded + cached)"
  ]
}

File tersimpan: data/interim/preprocessed_comments_kopdes.parquet (118,778 baris)


### Kesimpulan Tahap Preprocessing

Seluruh langkah wajib brief telah diterapkan dan tervalidasi:
`Case Folding -> Cleaning -> Remove URL -> Remove Mention -> Remove Hashtag -> Remove Emoji ->
Remove Number -> Remove Punctuation -> Tokenization -> Stopword Removal -> Stemming`

- **118.778 baris** akhir siap untuk pelabelan sentimen (dari 120.707 input, -1,6% karena residu
  kosong pasca-cleaning yang terdokumentasi).
- Stemming dioptimasi dengan **frequency-thresholding + disk caching** -- keputusan rekayasa
  performa yang terdokumentasi jelas, mencakup 95,37% dari seluruh kemunculan token.
- Stopword removal formal + kolokial disesuaikan untuk konteks Bahasa Indonesia informal media
  sosial.